In [14]:
import os, sys, time, signal, subprocess, urllib.request, urllib.error

VENV_PYTHON = "/content/venv/bin/python"

RECOVERY_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
RECOVERY_PORT = 8000
RECOVERY_LOG = "/content/server.log"

_R_TRANSFORMERS = "4.46.*"
_R_ACCELERATE = "1.1.*"
_R_NEED_AWQ = False
_R_VLLM = "0.6.*"; _R_HTTPX = "0.27.*"; _R_OPENAI = "1.54.*"

RECOVERY_ARGS = {
    "--model": RECOVERY_MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(RECOVERY_PORT),
}

subprocess.run(["pkill", "-f", "vllm.entrypoints.openai.api_server"], check=False)
time.sleep(2)

# vLLM goes in the 3.10 venv (it has no py3.13 wheel)
subprocess.run([VENV_PYTHON, "-m", "pip", "install", "-q",
                f"vllm=={_R_VLLM}", f"transformers=={_R_TRANSFORMERS}",
                f"accelerate=={_R_ACCELERATE}"]
               + (["autoawq==0.2.9"] if _R_NEED_AWQ else []),
               check=True)

# httpx/openai go in the notebook kernel (3.13) - the client side, not the server
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                f"httpx=={_R_HTTPX}", f"openai=={_R_OPENAI}"],
               check=True)
print("pins reinstalled")

_r_cmd = [VENV_PYTHON, "-m", "vllm.entrypoints.openai.api_server"]
for k, v in RECOVERY_ARGS.items():
    _r_cmd += [k] if v is None else [k, str(v)]
_r_logf = open(RECOVERY_LOG, "wb")
server = subprocess.Popen(_r_cmd, stdout=_r_logf, stderr=subprocess.STDOUT,
                          start_new_session=True)
print(f"relaunched server pid {server.pid}, logging to {RECOVERY_LOG}")

_deadline = time.time() + 300
while time.time() < _deadline:
    try:
        with urllib.request.urlopen(
                f"http://localhost:{RECOVERY_PORT}/v1/models", timeout=5) as r:
            if r.status == 200:
                print("RECOVERED: server healthy. continue from your last step.")
                break
    except (urllib.error.URLError, ConnectionError, OSError):
        pass
    time.sleep(3)
else:
    print("recovery timed out. last 30 log lines:")
    try:
        with open(RECOVERY_LOG, errors="replace") as fh:
            print("".join(fh.readlines()[-30:]))
    except FileNotFoundError:
        print("(no log file)")

pins reinstalled
relaunched server pid 21371, logging to /content/server.log
RECOVERED: server healthy. continue from your last step.


In [9]:
import sys
print(sys.version)

3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]


In [10]:
!sudo apt-get update -y
!sudo apt-get install -y python3.10 python3.10-venv python3.10-dev
!python3.10 -m venv /content/venv
!/content/venv/bin/python -m pip install --upgrade pip

VENV_PYTHON = "/content/venv/bin/python"

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,915 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,180 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.8 MB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy

In [11]:
import subprocess

VLLM_PIN = "0.6.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"

subprocess.run(
    [VENV_PYTHON, "-m", "pip", "install",
     f"vllm=={VLLM_PIN}", f"transformers=={TRANSFORMERS_PIN}",
     f"accelerate=={ACCELERATE_PIN}"],
    check=True
)
print("vLLM installed inside the 3.10 venv")

vLLM installed inside the 3.10 venv


In [15]:
import sys, subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "httpx==0.27.*", "openai==1.54.*"],
    check=True
)
print("client-side pins installed in the notebook kernel")

client-side pins installed in the notebook kernel


In [16]:
# Prediction card - fill by hand BEFORE running anything
static_scaling_1_to_8 = 74.6 / 28.2
predicted_vllm_scaling_1_to_8 = 4.0
expect_vllm_larger_than_static = True

prediction = {
    "static_scaling_1_to_8": static_scaling_1_to_8,
    "predicted_vllm_scaling_1_to_8": predicted_vllm_scaling_1_to_8,
    "expect_vllm_larger_than_static": expect_vllm_larger_than_static,
}

print(round(prediction["static_scaling_1_to_8"], 2), "x static scaling (from Monday)")
print("your prediction:", prediction["predicted_vllm_scaling_1_to_8"])

2.65 x static scaling (from Monday)
your prediction: 4.0


In [20]:
import subprocess

VENV_PYTHON = "/content/venv/bin/python"

VLLM_PIN = "0.6.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = [VENV_PYTHON, "-m", "pip", "install", "-q", *specs]
    print("installing (venv):", " ".join(specs))
    subprocess.run(cmd, check=True)

pip_install(
    f"vllm=={VLLM_PIN}",
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"httpx=={HTTPX_PIN}",
    f"openai=={OPENAI_PIN}",
)
print("serving pins installed inside the venv")

installing (venv): vllm==0.6.* transformers==4.46.* accelerate==1.1.* httpx==0.27.* openai==1.54.*
serving pins installed inside the venv


In [19]:
import os
print(os.path.exists("/content/venv/bin/python"))

True


In [21]:
import os, signal, subprocess

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000
SERVER_LOG = "/content/server.log"

SERVER_ARGS = {
    "--model": MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
}

def build_cmd(args: dict) -> list:
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        if v is None:
            cmd.append(k)
        else:
            cmd += [k, str(v)]
    return cmd

def launch_server(args: dict = None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)
    print("launching:", " ".join(cmd))
    logf = open(SERVER_LOG, "wb")
    proc = subprocess.Popen(
        cmd, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True,
    )
    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server()

launching: /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000
server pid 27078, logging to /content/server.log


In [22]:
import time, urllib.request, urllib.error

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(f"server healthy after about {waited}s: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass
        time.sleep(interval_s)
    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())
    print("server did not come up. common causes: model still downloading "
          "(rerun this cell), OOM at load (lower --gpu-memory-utilization to "
          "0.80), or a bad flag (bf16 on sm75; use --dtype half).")
    return False

healthy = wait_for_health()

server healthy after about 0s: http://localhost:8000/v1/models -> 200


In [23]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
r = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[{"role": "user", "content": "In one sentence, what is a GPU?"}],
)
print(r.choices[0].message.content)

A GPU, or Graphics Processing Unit, is a specialized processor designed to accelerate computations involved in rendering graphics and video content on electronic devices.


In [24]:
from google.colab import files
uploaded = files.upload()
import json
baseline = json.load(open("baselines.json"))
print("baseline batch tokens/s:", baseline["batch"])

Saving baselines.json to baselines.json
baseline batch tokens/s: {'1': 28.2, '4': 35.0, '8': 74.6}


In [27]:
# Async A/B client for Lab W3D3 (engine swap).
import asyncio
import time
import httpx

FIXED_PROMPTS = [
    "In one sentence, what is a GPU?",
    "List three reasons decode is memory-bound.",
    "Explain the KV cache to a new ops engineer in two sentences.",
    "What does continuous batching change versus static batching?",
    "Give a one-line definition of tokens per second.",
    "Why does a longer prompt increase time to first token?",
    "Name two things quantisation trades away for smaller memory.",
    "Summarise what an inference server does in three short bullets.",
]

QUEUE = [32, 32, 32, 256] * 6
MAX_TOKENS = 128
WARMUP = 4


async def _one_request(client, base_url, model, prompt, max_tokens=MAX_TOKENS):
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": 0.0,
        "stream": False,
    }
    r = await client.post(f"{base_url}/chat/completions", json=payload)
    r.raise_for_status()
    body = r.json()
    usage = body.get("usage", {})
    ct = usage.get("completion_tokens")
    if ct is None:
        ct = len(body["choices"][0]["message"]["content"].split())
    return ct


async def _run_level(client, base_url, model, prompts, concurrency, total_requests):
    sem = asyncio.Semaphore(concurrency)
    counts = []

    async def guarded(prompt, max_tokens):
        async with sem:
            return await _one_request(client, base_url, model, prompt, max_tokens)

    tasks = [asyncio.create_task(guarded(prompts[i % len(prompts)],
                                         QUEUE[i % len(QUEUE)]))
             for i in range(total_requests)]
    t0 = time.time()
    for coro in asyncio.as_completed(tasks):
        counts.append(await coro)
    dt = time.time() - t0
    total_tokens = sum(counts)
    return {
        "concurrency": concurrency,
        "requests": total_requests,
        "tokens_per_s": round(total_tokens / dt, 1),
        "wall_s": round(dt, 3),
    }


async def run_sweep(base_url, model, prompts=FIXED_PROMPTS,
                    concurrencies=(1, 4, 8), requests_per_level=24):
    results = []
    async with httpx.AsyncClient(timeout=120.0) as client:
        await asyncio.gather(*[
            _one_request(client, base_url, model, prompts[i % len(prompts)])
            for i in range(WARMUP)
        ])
        for c in concurrencies:
            level = await _run_level(client, base_url, model, prompts, c,
                                     requests_per_level)
            print("level:", level)
            results.append(level)
    return results

In [28]:
prompts = FIXED_PROMPTS
vllm_measured = await run_sweep(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    prompts=prompts,
    concurrencies=[1, 4, 8],
)
for level in vllm_measured:
    print(level)

level: {'concurrency': 1, 'requests': 24, 'tokens_per_s': 58.8, 'wall_s': 23.634}
level: {'concurrency': 4, 'requests': 24, 'tokens_per_s': 169.5, 'wall_s': 8.196}
level: {'concurrency': 8, 'requests': 24, 'tokens_per_s': 230.7, 'wall_s': 6.02}
{'concurrency': 1, 'requests': 24, 'tokens_per_s': 58.8, 'wall_s': 23.634}
{'concurrency': 4, 'requests': 24, 'tokens_per_s': 169.5, 'wall_s': 8.196}
{'concurrency': 8, 'requests': 24, 'tokens_per_s': 230.7, 'wall_s': 6.02}


In [29]:
import json

def tokps_at(level_list, c):
    return next(x["tokens_per_s"] for x in level_list if x["concurrency"] == c)

vllm_by_c = {x["concurrency"]: x["tokens_per_s"] for x in vllm_measured}
base_by_c = {int(k): v for k, v in baseline["batch"].items()}

speedup = {c: round(vllm_by_c[c] / base_by_c[c], 2)
           for c in vllm_by_c if c in base_by_c}

report = {
    "baseline": base_by_c,
    "vllm": vllm_by_c,
    "speedup_by_concurrency": speedup,
    "predicted_speedup": prediction["predicted_vllm_scaling_1_to_8"],  # من Cell 0
}

with open("ab_report.json", "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))

{
  "baseline": {
    "1": 28.2,
    "4": 35.0,
    "8": 74.6
  },
  "vllm": {
    "1": 58.8,
    "4": 169.5,
    "8": 230.7
  },
  "speedup_by_concurrency": {
    "1": 2.09,
    "4": 4.84,
    "8": 3.09
  },
  "predicted_speedup": 4.0
}


In [30]:
static_scaling = base_by_c[8] / base_by_c[1]
vllm_scaling   = vllm_by_c[8] / vllm_by_c[1]

print(f"static batching scales {static_scaling:.2f}x")
print(f"vLLM scales {vllm_scaling:.2f}x")
print(f"continuous batching is worth {vllm_scaling / static_scaling:.2f}x of scaling")

# سلّم concurrency-8 tokens/s على لوحة التقدم تحت اسم فريقك:
print("submit this number:", vllm_by_c[8])

static batching scales 2.65x
vLLM scales 3.92x
continuous batching is worth 1.48x of scaling
submit this number: 230.7


In [31]:
def shutdown_server(proc=None, port=PORT):
    try:
        proc = server if proc is None else proc
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        print(f"sent SIGTERM to process group of pid {proc.pid}")
    except (ProcessLookupError, NameError):
        print("no server process to kill")
    time.sleep(3)
    try:
        with urllib.request.urlopen(f"http://localhost:{port}/v1/models", timeout=2):
            print(f"WARNING: port {port} still answering; something is still up")
    except (urllib.error.URLError, ConnectionError, OSError):
        print(f"port {port} is free")

shutdown_server()

sent SIGTERM to process group of pid 27078


In [32]:
# Green-check verifier for Lab W3D3 (engine swap).
import json, os


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    path = "ab_report.json"
    if not os.path.exists(path):
        fail(f"{path} not found; write it in Cell 5")
    try:
        with open(path) as fh:
            report = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"{path} is not valid JSON: {exc}")

    for key in ("baseline", "vllm", "speedup_by_concurrency"):
        if key not in report:
            fail(f"{path} missing key: {key}")

    baseline = report["baseline"]
    vllm = report["vllm"]
    speedup = report["speedup_by_concurrency"]

    if not isinstance(baseline, dict) or not baseline:
        fail("baseline must be a non-empty object (from Monday's baselines.json)")
    if not isinstance(vllm, dict) or not vllm:
        fail("vllm must be a non-empty object of measured throughput")
    if not isinstance(speedup, dict) or not speedup:
        fail("speedup_by_concurrency must be a non-empty object")

    def get_c(d, c):
        for k, v in d.items():
            if str(k) == str(c):
                return v
        return None

    base8 = get_c(baseline, 8)
    vllm8 = get_c(vllm, 8)
    if base8 is None:
        fail("baseline has no concurrency-8 (batch-8) number")
    if vllm8 is None:
        fail("vllm has no concurrency-8 number")
    if not isinstance(base8, (int, float)) or not isinstance(vllm8, (int, float)):
        fail("concurrency-8 throughput values must be numbers")

    if not vllm8 > base8:
        fail(f"vllm concurrency-8 throughput ({vllm8}) not above baseline "
             f"batch-8 ({base8}); the engine swap should win here")

    s8 = get_c(speedup, 8)
    if s8 is None or not isinstance(s8, (int, float)):
        fail("speedup_by_concurrency has no numeric value at concurrency 8")
    expected = vllm8 / base8
    if abs(s8 - expected) > 0.1:
        fail(f"speedup at 8 ({s8}) does not match vllm/baseline "
             f"({expected:.2f}); recompute it")

    print(f"baseline batch-8: {base8}, vllm concurrency-8: {vllm8}")
    print(f"speedup at 8: {s8}x")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    try:
        get_ipython()
    except NameError:
        raise SystemExit(1)

baseline batch-8: 74.6, vllm concurrency-8: 230.7
speedup at 8: 3.09x
GREEN CHECK: PASS
